In [1]:
!pip install huggingface_hub[hf_xet] pandas transformers torch

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 14.4 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import re

# Load data
df = pd.read_csv('../data/demo.csv')
texts = df['text'].tolist()

# Preprocess
def clean_text(s: str):
    # Replace all curly double quotes with "
    s = re.sub(r'[“”]', '"', s)
     
    # Replace all curly single quotes with '
    s = re.sub(r"[‘’]", "'", s)
    
    # Replace en dash and em dash with hyphen
    s = re.sub(r"[–—]", "-", s)

    # Only retain alphanumeric, whitespace characters, single and double quotes, and hyphens
    s = re.sub(pattern=rf"[^a-zA-Z0-9\s\-\'\"]", repl="", string=s, flags=re.IGNORECASE)

    # Remove extra whitespaces
    s = re.sub(pattern=r"\s+", repl=" ", string=s).strip()

    return s

def preprocess(text: str):
    return clean_text(text)

# Load pretrained model and tokenizer
tokenizer = BertTokenizer.from_pretrained("njlr/cs180-project")
model = BertForSequenceClassification.from_pretrained("njlr/cs180-project")
model.eval()

# Tokenize
cleaned_text = [preprocess(text) for text in texts]
inputs = tokenizer(cleaned_text, padding=True, truncation=True, return_tensors='pt')

# Predict
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).tolist()

# Save predictions
df['predictions'].to_csv('../predictions/bert_predictions.csv', index=False)

C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
